<a href="https://colab.research.google.com/github/Ashraf310120/Python_Assignments_DS_/blob/main/Document_Inelligence_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project 1 :-**
#   ** Document Intelligence RAG System**

In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [ ]:
# ============================================================
# STEP 1: LOAD THE LANGUAGE MODEL ON T4 GPU
# ============================================================

import torch
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device=0,
    model_kwargs={
        "torch_dtype": torch.float16
    }
)

print("Model loaded on:", torch.cuda.get_device_name(0))
print("GPU model is ready!")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded on: Tesla T4
GPU model is ready!


In [ ]:
# # ============================================================
# # FIX: INSTALL PDF READER AND RESTORE YOUR PDF starts by here
# # ============================================================

# %pip install -q pypdf

# from google.colab import files
# from pypdf import PdfReader

# # Upload your resume PDF again.
# uploaded = files.upload()

# pdf_files = [
#     name for name in uploaded
#     if name.lower().endswith(".pdf")
# ]

# if not pdf_files:
#     raise ValueError("Please upload a PDF file.")

# pdf_path = pdf_files[0]
# reader = PdfReader(pdf_path)

# pdf_text = "\n".join(
#     f"[Page {i + 1}]\n{page.extract_text() or ''}"
#     for i, page in enumerate(reader.pages)
# )

# if not pdf_text.strip():
#     raise ValueError(
#         "No readable text extracted. This PDF may be scanned."
#     )

# print("PDF restored successfully!")
# print("Pages:", len(reader.pages))
# print("Extracted characters:", len(pdf_text))
# print("\nPreview:")
# print(pdf_text[:500])
# ============================================================
# SWITCH DOCUMENTS WITHOUT RELOADING QWEN
# Supported: PDF, TXT, DOCX, MD, CSV, JSON
# ============================================================

import os
import sys
import subprocess
import importlib.util
import numpy as np
import faiss

from google.colab import files
from pypdf import PdfReader

# Check that the existing models are loaded.
if "embedding_model" not in globals() or "generator" not in globals():
    raise RuntimeError(
        "Please run the original RAG setup first."
    )

# Upload a new document.
uploaded = files.upload()

if not uploaded:
    raise ValueError("No file uploaded.")

new_path = next(iter(uploaded))
extension = os.path.splitext(new_path)[1].lower()

# 1. Extract text according to the file type.
if extension == ".pdf":
    reader = PdfReader(new_path)
    extracted_pages = []

    for page_number, page in enumerate(reader.pages, start=1):
        page_content = page.extract_text() or ""

        if page_content.strip():
            extracted_pages.append(
                f"[Page {page_number}]\n{page_content}"
            )

    new_text = "\n\n".join(extracted_pages)

elif extension == ".docx":
    if importlib.util.find_spec("docx") is None:
        subprocess.check_call([
            sys.executable, "-m", "pip",
            "install", "-q", "python-docx"
        ])

    from docx import Document

    document = Document(new_path)
    new_text = "\n".join(
        paragraph.text
        for paragraph in document.paragraphs
        if paragraph.text.strip()
    )

elif extension in {".txt", ".md", ".csv", ".json"}:
    with open(new_path, "r", encoding="utf-8", errors="replace") as file:
        new_text = file.read()

else:
    raise ValueError(
        "Unsupported format. Use PDF, TXT, DOCX, MD, CSV or JSON."
    )

if not new_text.strip():
    raise ValueError(
        "No readable text found. Scanned PDFs may require OCR."
    )

# 2. Create text chunks.
def create_chunks(text, chunk_words=150, overlap_words=30):
    words = text.split()
    chunks = []
    step = chunk_words - overlap_words

    for start in range(0, len(words), step):
        chunk = " ".join(words[start:start + chunk_words])

        if chunk.strip():
            chunks.append(chunk)

        if start + chunk_words >= len(words):
            break

    return chunks

pdf_text = new_text
pdf_chunks = create_chunks(pdf_text)

# 3. Embed the new document using the existing embedding model.
pdf_embeddings = embedding_model.encode(
    pdf_chunks,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype(np.float32)

# 4. Rebuild the FAISS index.
pdf_index = faiss.IndexFlatIP(pdf_embeddings.shape[1])
pdf_index.add(pdf_embeddings)

# 5. Rebuild keyword search.
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    sublinear_tf=True
)

tfidf_matrix = tfidf_vectorizer.fit_transform(pdf_chunks)

print("\nNew document loaded successfully!")
print("File:", new_path)
print("Characters:", len(pdf_text))
print("Chunks:", len(pdf_chunks))
print("Index rebuilt. Ready for questions!")

Saving RAG_Demo_Long_Sample_Report (1).pdf to RAG_Demo_Long_Sample_Report (1).pdf


Batches:   0%|          | 0/2 [00:00<?, ?it/s]


New document loaded successfully!
File: RAG_Demo_Long_Sample_Report (1).pdf
Characters: 31434
Chunks: 39
Index rebuilt. Ready for questions!


In [ ]:
# ============================================================
# STEP 2: REBUILD THE PDF SEARCH INDEX
# ============================================================

%pip install -q sentence-transformers faiss-cpu scikit-learn

import re
import numpy as np
import torch
import faiss

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Verify the uploaded PDF text is available.
if "pdf_text" not in globals() or not pdf_text.strip():
    raise ValueError("PDF text is missing. Check the PDF extraction cell.")

# 2. Split the PDF into manageable chunks.
def build_chunks(text, max_chars=750):
    lines = [
        re.sub(r"\s+", " ", line).strip()
        for line in text.splitlines()
        if line.strip()
    ]

    chunks = []
    current = ""

    for line in lines:
        words = line.split()
        pieces = []
        piece = ""

        for word in words:
            if piece and len(piece) + len(word) + 1 > max_chars:
                pieces.append(piece)
                piece = word
            else:
                piece = f"{piece} {word}".strip()

        if piece:
            pieces.append(piece)

        for part in pieces:
            if current and len(current) + len(part) + 1 > max_chars:
                chunks.append(current)
                current = part
            else:
                current = f"{current}\n{part}".strip()

    if current:
        chunks.append(current)

    return chunks


pdf_chunks = build_chunks(pdf_text)

if not pdf_chunks:
    raise ValueError("No readable PDF chunks were created.")

# 3. Load the lightweight embedding model on the GPU.
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# 4. Generate embeddings.
pdf_embeddings = embedding_model.encode(
    pdf_chunks,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype(np.float32)

# 5. Create a FAISS similarity-search index.
pdf_index = faiss.IndexFlatIP(pdf_embeddings.shape[1])
pdf_index.add(pdf_embeddings)

# 6. Create keyword-search data.
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    sublinear_tf=True
)

tfidf_matrix = tfidf_vectorizer.fit_transform(pdf_chunks)

print("\nPDF index rebuilt successfully!")
print("Pages/text available:", len(pdf_text))
print("Chunks:", len(pdf_chunks))
print("Indexed vectors:", pdf_index.ntotal)
print("GPU available:", torch.cuda.is_available())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]


PDF index rebuilt successfully!
Pages/text available: 31434
Chunks: 44
Indexed vectors: 44
GPU available: True


In [ ]:
# ============================================================
# STEP 2: REBUILD THE PDF SEARCH INDEX
# ============================================================

%pip install -q sentence-transformers faiss-cpu scikit-learn

import re
import numpy as np
import torch
import faiss

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Verify the uploaded PDF text is available.
if "pdf_text" not in globals() or not pdf_text.strip():
    raise ValueError("PDF text is missing. Check the PDF extraction cell.")

# 2. Split the PDF into manageable chunks.
def build_chunks(text, max_chars=750):
    lines = [
        re.sub(r"\s+", " ", line).strip()
        for line in text.splitlines()
        if line.strip()
    ]

    chunks = []
    current = ""

    for line in lines:
        words = line.split()
        pieces = []
        piece = ""

        for word in words:
            if piece and len(piece) + len(word) + 1 > max_chars:
                pieces.append(piece)
                piece = word
            else:
                piece = f"{piece} {word}".strip()

        if piece:
            pieces.append(piece)

        for part in pieces:
            if current and len(current) + len(part) + 1 > max_chars:
                chunks.append(current)
                current = part
            else:
                current = f"{current}\n{part}".strip()

    if current:
        chunks.append(current)

    return chunks


pdf_chunks = build_chunks(pdf_text)

if not pdf_chunks:
    raise ValueError("No readable PDF chunks were created.")

# 3. Load the lightweight embedding model on the GPU.
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# 4. Generate embeddings.
pdf_embeddings = embedding_model.encode(
    pdf_chunks,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype(np.float32)

# 5. Create a FAISS similarity-search index.
pdf_index = faiss.IndexFlatIP(pdf_embeddings.shape[1])
pdf_index.add(pdf_embeddings)

# 6. Create keyword-search data.
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    sublinear_tf=True
)

tfidf_matrix = tfidf_vectorizer.fit_transform(pdf_chunks)

print("\nPDF index rebuilt successfully!")
print("Pages/text available:", len(pdf_text))
print("Chunks:", len(pdf_chunks))
print("Indexed vectors:", pdf_index.ntotal)
print("GPU available:", torch.cuda.is_available())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]


PDF index rebuilt successfully!
Pages/text available: 31434
Chunks: 44
Indexed vectors: 44
GPU available: True


In [ ]:
# ============================================================
# STEP 4: OPTIMIZED PDF RAG QUESTION ANSWERING
# General-purpose | GPU | No API key
# ============================================================

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def ask_pdf(question):
    if not question or not question.strip():
        print("Please enter a question.")
        return

    # 1. Convert the question into an embedding.
    query_vector = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)[0]

    # 2. Retrieve relevant chunks using semantic and keyword search.
    semantic_scores = pdf_embeddings @ query_vector

    lexical_scores = cosine_similarity(
        tfidf_vectorizer.transform([question]),
        tfidf_matrix
    ).ravel()

    score_range = np.ptp(semantic_scores)

    if score_range > 1e-8:
        semantic_scores = (
            semantic_scores - semantic_scores.min()
        ) / score_range
    else:
        semantic_scores = np.zeros_like(semantic_scores)

    combined_scores = (
        0.6 * semantic_scores + 0.4 * lexical_scores
    )

    # 3. Select the best chunks within a compact context budget.
    ranked_ids = np.argsort(combined_scores)[::-1]
    selected_chunks = []
    used_ids = set()
    context_length = 0
    max_context_chars = 2800

    for idx in ranked_ids:
        idx = int(idx)
        chunk = pdf_chunks[idx].strip()

        if not chunk or idx in used_ids:
            continue

        if context_length + len(chunk) > max_context_chars:
            continue

        selected_chunks.append(chunk)
        used_ids.add(idx)
        context_length += len(chunk)

        if len(selected_chunks) == 5:
            break

    if not selected_chunks:
        print("No relevant document text retrieved.")
        return

    context = "\n\n".join(selected_chunks)

    # 4. Ask Qwen to answer directly from the retrieved context.
    messages = [
        {
            "role": "system",
            "content": (
                "Answer using only the supplied document excerpts. "
                "Give a direct, complete answer. Include all relevant "
                "items when asked for lists or counts. Preserve exact "
                "names and numbers. Do not invent missing facts. "
                "If the document does not provide the answer, say "
                "'Not specified in the document.' Return only the answer."
            )
        },
        {
            "role": "user",
            "content": (
                f"Document excerpts:\n{context}\n\n"
                f"Question: {question}"
            )
        }
    ]

    prompt = generator.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # 5. Generate a concise answer on the GPU.
    result = generator(
        prompt,
        max_new_tokens=64,
        do_sample=False,
        return_full_text=False,
        pad_token_id=generator.tokenizer.eos_token_id
    )

    answer = result[0]["generated_text"].strip()
    print(answer if answer else "No answer generated.")


print("PDF RAG function restored successfully!")

PDF RAG function restored successfully!


In [ ]:

# ============================================================
# IMPROVED RAG: RETRIEVE MORE CONTEXT FOR BROAD QUESTIONS
# ============================================================

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def ask_pdf(question):
    if not question or not question.strip():
        print("Please enter a question.")
        return

    # 1. Search by semantic meaning.
    query_vector = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)[0]

    semantic_scores = pdf_embeddings @ query_vector

    # Normalize semantic scores.
    score_range = np.ptp(semantic_scores)

    if score_range > 1e-8:
        semantic_scores = (
            semantic_scores - semantic_scores.min()
        ) / score_range
    else:
        semantic_scores = np.zeros_like(semantic_scores)

    # 2. Search for matching words and phrases.
    lexical_scores = cosine_similarity(
        tfidf_vectorizer.transform([question]),
        tfidf_matrix
    ).ravel()

    # 3. Combine the two retrieval methods.
    combined_scores = (
        0.60 * semantic_scores
        + 0.40 * lexical_scores
    )

    # Retrieve more candidates than the old top-four approach.
    ranked_ids = np.argsort(combined_scores)[::-1]
    selected_ids = set()

    for idx in ranked_ids[:10]:
        idx = int(idx)

        # Include neighboring chunks for additional context.
        for neighbor in (idx - 1, idx, idx + 1):
            if 0 <= neighbor < len(pdf_chunks):
                selected_ids.add(neighbor)

    # 4. Prioritize relevant chunks while respecting context size.
    prioritized_ids = sorted(
        selected_ids,
        key=lambda i: combined_scores[i],
        reverse=True
    )

    chosen_ids = []
    used_chars = 0
    max_context_chars = 8500

    for idx in prioritized_ids:
        chunk = pdf_chunks[idx].strip()

        if not chunk:
            continue

        if used_chars + len(chunk) > max_context_chars:
            continue

        chosen_ids.append(idx)
        used_chars += len(chunk)

    # Restore document order to help the model interpret timelines.
    chosen_ids.sort()
    context = "\n\n".join(
        pdf_chunks[idx] for idx in chosen_ids
    )

    if not context:
        print("No relevant document context found.")
        return

    # 5. Ask Qwen to answer completely using the retrieved evidence.
    messages = [
        {
            "role": "system",
            "content": (
                "Answer only from the document excerpts. "
                "For questions asking for all dates, milestones, "
                "figures, or items, provide a complete list of every "
                "relevant item present in the supplied context. "
                "Preserve dates and their corresponding events. "
                "Do not invent missing information. "
                "Return only the answer."
            )
        },
        {
            "role": "user",
            "content": (
                f"Document excerpts:\n{context}\n\n"
                f"Question: {question}"
            )
        }
    ]

    prompt = generator.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    result = generator(
        prompt,
        max_new_tokens=200,
        do_sample=False,
        return_full_text=False,
        pad_token_id=generator.tokenizer.eos_token_id
    )

    print(result[0]["generated_text"].strip())


print("Improved retrieval function is ready.")


Improved retrieval function is ready.


In [ ]:
# ============================================================
# FIXED RAG FUNCTION: COMPLETE ANSWERS WITH BETTER CONTEXT
# ============================================================

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def ask_pdf(question):

    if not question or not question.strip():
        print("Please enter a question.")
        return

    # STEP 1: Embed the question.
    query_vector = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)[0]

    # STEP 2: Semantic similarity search.
    semantic_scores = pdf_embeddings @ query_vector

    score_range = np.ptp(semantic_scores)

    if score_range > 1e-8:
        semantic_scores = (
            semantic_scores - semantic_scores.min()
        ) / score_range
    else:
        semantic_scores = np.zeros_like(semantic_scores)

    # STEP 3: Keyword search.
    lexical_scores = cosine_similarity(
        tfidf_vectorizer.transform([question]),
        tfidf_matrix
    ).ravel()

    # STEP 4: Combine semantic and keyword scores.
    combined_scores = (
        0.60 * semantic_scores
        + 0.40 * lexical_scores
    )

    # STEP 5: Select relevant chunks and neighboring context.
    ranked_ids = np.argsort(combined_scores)[::-1]
    selected_ids = set()

    for idx in ranked_ids[:10]:
        idx = int(idx)

        for neighbor in (idx - 1, idx, idx + 1):
            if 0 <= neighbor < len(pdf_chunks):
                selected_ids.add(neighbor)

    # Preserve document order.
    ordered_ids = sorted(
        selected_ids,
        key=lambda i: combined_scores[i],
        reverse=True
    )

    chosen_ids = []
    used_chars = 0
    max_context_chars = 8500

    for idx in ordered_ids:
        chunk = pdf_chunks[idx].strip()

        if not chunk:
            continue

        if used_chars + len(chunk) > max_context_chars:
            continue

        chosen_ids.append(idx)
        used_chars += len(chunk)

    # Restore original document order for timelines.
    chosen_ids.sort()

    context = "\n\n".join(
        pdf_chunks[idx] for idx in chosen_ids
    )

    if not context:
        print("No relevant document context found.")
        return

    # STEP 6: Build the complete prompt.
    messages = [
        {
            "role": "system",
            "content": (
                "Answer using only the supplied document excerpts. "
                "Give a direct and complete answer. For questions "
                "asking for all dates, milestones, figures, or items, "
                "include every relevant item present in the context. "
                "Preserve dates exactly as written. If the document "
                "provides only a month or date range, do not invent "
                "an exact day. Keep each date paired with its event. "
                "Do not invent facts or combine separate milestones. "
                "If information is absent, say it is not specified. "
                "Return only the answer."
            )
        },
        {
            "role": "user",
            "content": (
                f"DOCUMENT EXCERPTS:\n{context}\n\n"
                f"QUESTION:\n{question}"
            )
        }
    ]

    # STEP 7: Apply Qwen's chat template.
    prompt = generator.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # STEP 8: Generate a longer answer on the GPU.
    result = generator(
        prompt,
        max_new_tokens=350,
        do_sample=False,
        return_full_text=False,
        pad_token_id=generator.tokenizer.eos_token_id
    )

    answer = result[0]["generated_text"].strip()

    print(answer if answer else "No answer generated.")


print("Corrected RAG function is ready!")

Corrected RAG function is ready!


In [ ]:
# ============================================================
# SOURCE-GROUNDED RAG
# Exact date extraction + normal Q&A through Qwen
# ============================================================

import re
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


def extract_document_dates(text):
    """Extract dates and their associated text directly from a document."""

    lines = [line.strip() for line in text.splitlines() if line.strip()]

    months = (
        r"Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|"
        r"May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|"
        r"Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?"
    )

    full_date = re.compile(
        rf"^\d{{1,2}}\s+(?:{months})\s+\d{{4}}$",
        re.IGNORECASE
    )

    month_period = re.compile(
        rf"^(?:{months})(?:\s*[-–]\s*(?:{months}))?\s+\d{{4}}$",
        re.IGNORECASE
    )

    entries = []
    status_words = {"completed", "planned", "in progress"}

    for i, line in enumerate(lines):

        # Exact day-month-year entries, such as 6 Jan 2025.
        if full_date.fullmatch(line):
            event = ""

            for following in lines[i + 1:i + 4]:
                if following.lower() in status_words:
                    continue

                if following.startswith("[Page "):
                    continue

                event = following
                break

            if event:
                entries.append(f"{line} — {event}")

        # Month/year periods, such as Oct-Nov 2025.
        elif month_period.fullmatch(line):
            preceding = []
            j = i - 1

            # Find a nearby phase or milestone heading.
            while j >= 0 and len(preceding) < 3:
                candidate = lines[j]

                if candidate.startswith("[Page "):
                    break

                if candidate.lower() in {
                    "roadmap", "timeline", "target date",
                    "date", "milestone"
                }:
                    break

                preceding.append(candidate)

                if candidate.lower().startswith("phase "):
                    break

                j -= 1

            event = " ".join(reversed(preceding)).strip()

            if event:
                entries.append(f"{line} — {event}")

    # Preserve document order and remove duplicates.
    return list(dict.fromkeys(entries))


def ask_pdf(question):
    if not question or not question.strip():
        print("Please enter a question.")
        return

    q = question.lower()

    # For date/timeline questions, extract source text directly.
    # Do not let the language model invent missing dates.
    date_words = (
        "date", "dates", "timeline", "milestone",
        "planned future", "roadmap"
    )

    if any(word in q for word in date_words):
        entries = extract_document_dates(pdf_text)

        if entries:
            print("\n".join(entries))
        else:
            print("No standalone dates or milestones were found.")
        return

    # Normal RAG retrieval for all other questions.
    query_vector = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)[0]

    semantic_scores = pdf_embeddings @ query_vector
    lexical_scores = cosine_similarity(
        tfidf_vectorizer.transform([question]),
        tfidf_matrix
    ).ravel()

    score_range = np.ptp(semantic_scores)

    if score_range > 1e-8:
        semantic_scores = (
            semantic_scores - semantic_scores.min()
        ) / score_range
    else:
        semantic_scores = np.zeros_like(semantic_scores)

    scores = 0.6 * semantic_scores + 0.4 * lexical_scores
    ranked_ids = np.argsort(scores)[::-1]

    selected_ids = set()

    for idx in ranked_ids[:8]:
        idx = int(idx)

        for neighbor in (idx - 1, idx, idx + 1):
            if 0 <= neighbor < len(pdf_chunks):
                selected_ids.add(neighbor)

    context_parts = []
    total_chars = 0

    for idx in sorted(selected_ids):
        chunk = pdf_chunks[idx].strip()

        if total_chars + len(chunk) > 6000:
            continue

        context_parts.append(chunk)
        total_chars += len(chunk)

    context = "\n\n".join(context_parts)

    messages = [
        {
            "role": "system",
            "content": (
                "Answer using only the supplied document excerpts. "
                "Give a direct, complete answer. Preserve names, "
                "numbers, and facts. Do not invent information. "
                "If something is not stated, say so. Return only "
                "the answer without source labels."
            )
        },
        {
            "role": "user",
            "content": f"Document:\n{context}\n\nQuestion: {question}"
        }
    ]

    prompt = generator.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    result = generator(
        prompt,
        max_new_tokens=160,
        do_sample=False,
        return_full_text=False,
        pad_token_id=generator.tokenizer.eos_token_id
    )

    print(result[0]["generated_text"].strip())


print("Source-grounded RAG is ready.")

Source-grounded RAG is ready.


In [ ]:
# ============================================================
# IMPROVEMENT: MERGE DUPLICATE DATES
# No re-upload, no re-indexing, no model reload
# ============================================================

import re

# Preserve the current date extractor.
if "_original_date_extractor" not in globals():
    _original_date_extractor = extract_document_dates

def extract_document_dates(text):
    entries = _original_date_extractor(text)
    grouped = {}

    for entry in entries:
        parts = entry.split(" — ", 1)

        if len(parts) == 2:
            date_text, event = parts
        else:
            date_text, event = entry, ""

        key = re.sub(r"\s+", " ", date_text).strip().lower()

        if key not in grouped:
            grouped[key] = [date_text, []]

        if event and event not in grouped[key][1]:
            grouped[key][1].append(event)

    results = []

    for date_text, events in grouped.values():
        description = "; ".join(events)
        results.append(
            f"{date_text} — {description}" if description
            else date_text
        )

    return results

print("Duplicate-date handling enabled.")

Duplicate-date handling enabled.


In [ ]:
# ============================================================
# FINAL POLISH: SORT TIMELINE CHRONOLOGICALLY
# ============================================================

import re
from datetime import date, datetime

def extract_document_dates(text):
    entries = _original_date_extractor(text)
    grouped = {}

    for entry in entries:
        parts = entry.split(" — ", 1)
        date_text = parts[0].strip()
        event = parts[1].strip() if len(parts) > 1 else ""

        key = date_text.lower()

        if key not in grouped:
            grouped[key] = [date_text, []]

        if event and event not in grouped[key][1]:
            grouped[key][1].append(event)

    def sort_key(item):
        date_text = item[1][0]

        # Exact dates, e.g. 6 Oct 2025.
        match = re.match(
            r"(\d{1,2})\s+([A-Za-z]+)\s+(\d{4})",
            date_text
        )

        if match:
            day, month, year = match.groups()
            try:
                return date(int(year), datetime.strptime(
                    month[:3], "%b"
                ).month, int(day))
            except ValueError:
                pass

        # Month or month-range entries, e.g. Oct-Nov 2025.
        match = re.match(
            r"([A-Za-z]+)(?:[-–]([A-Za-z]+))?\s+(\d{4})",
            date_text
        )

        if match:
            month, _, year = match.groups()
            try:
                return date(
                    int(year),
                    datetime.strptime(month[:3], "%b").month,
                    1
                )
            except ValueError:
                pass

        return date.max

    ordered = sorted(grouped.values(), key=sort_key)

    return [
        f"{date_text} — {'; '.join(events)}"
        for date_text, events in ordered
    ]

print("Timeline sorting enabled.")

Timeline sorting enabled.


In [ ]:
# ============================================================
# FIX: SORT EXTRACTED DATES CHRONOLOGICALLY
# ============================================================

import re
from datetime import date, datetime

def extract_document_dates(text):
    entries = _original_date_extractor(text)
    grouped = {}

    # Remove duplicate dates and merge their descriptions.
    for entry in entries:
        parts = entry.split(" — ", 1)
        date_text = parts[0].strip()
        event = parts[1].strip() if len(parts) > 1 else ""

        key = date_text.lower()

        if key not in grouped:
            grouped[key] = [date_text, []]

        if event and event not in grouped[key][1]:
            grouped[key][1].append(event)

    def sort_key(item):
        # FIX: item[0] is the date; item[1] contains the events.
        date_text = item[0]

        # Exact dates, such as 6 Oct 2025.
        match = re.fullmatch(
            r"(\d{1,2})\s+([A-Za-z]+)\s+(\d{4})",
            date_text
        )

        if match:
            day, month, year = match.groups()
            try:
                parsed = datetime.strptime(
                    f"{day} {month[:3]} {year}", "%d %b %Y"
                ).date()
                return (parsed.year, parsed.month, parsed.day)
            except ValueError:
                pass

        # Month/year or month-range dates.
        match = re.fullmatch(
            r"([A-Za-z]+)(?:[-–]([A-Za-z]+))?\s+(\d{4})",
            date_text
        )

        if match:
            month, _, year = match.groups()
            try:
                parsed = datetime.strptime(
                    f"{month[:3]} {year}", "%b %Y"
                ).date()
                return (parsed.year, parsed.month, 1)
            except ValueError:
                pass

        return (9999, 12, 31)

    ordered = sorted(grouped.values(), key=sort_key)

    return [
        f"{date_text} — {'; '.join(events)}"
        for date_text, events in ordered
    ]


print("Date sorting fixed!")

Date sorting fixed!


In [ ]:
# ============================================================
# FINAL RAG FUNCTION
# Timeline extraction + general PDF question answering
# ============================================================

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


def ask_pdf(question):
    if not question or not question.strip():
        print("Please enter a question.")
        return

    q = question.lower().strip()

    # STEP 1: Route only timeline-related questions.
    timeline_terms = (
        "timeline",
        "milestone",
        "milestones",
        "roadmap",
        "planned future",
        "pilot dates",
        "rollout phases",
        "project dates",
        "project milestone dates",
    )

    if any(term in q for term in timeline_terms):
        entries = extract_document_dates(pdf_text)

        if entries:
            print("\n".join(entries))
        else:
            print("No standalone dates or milestones found.")
        return

    # STEP 2: Embed the question for general RAG retrieval.
    query_vector = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)[0]

    # STEP 3: Semantic similarity search.
    semantic_scores = pdf_embeddings @ query_vector
    score_range = np.ptp(semantic_scores)

    if score_range > 1e-8:
        semantic_scores = (
            semantic_scores - semantic_scores.min()
        ) / score_range
    else:
        semantic_scores = np.zeros_like(semantic_scores)

    # STEP 4: Keyword similarity search.
    lexical_scores = cosine_similarity(
        tfidf_vectorizer.transform([question]),
        tfidf_matrix
    ).ravel()

    # STEP 5: Combine search scores.
    combined_scores = (
        0.60 * semantic_scores
        + 0.40 * lexical_scores
    )

    ranked_ids = np.argsort(combined_scores)[::-1]
    selected_ids = set()

    # Include nearby chunks to preserve context.
    for idx in ranked_ids[:10]:
        idx = int(idx)

        for neighbor in (idx - 1, idx, idx + 1):
            if 0 <= neighbor < len(pdf_chunks):
                selected_ids.add(neighbor)

    # Keep the best-ranked chunks within the context budget.
    prioritized_ids = sorted(
        selected_ids,
        key=lambda i: combined_scores[i],
        reverse=True
    )

    chosen_ids = []
    used_chars = 0
    max_context_chars = 6000

    for idx in prioritized_ids:
        chunk = pdf_chunks[idx].strip()

        if not chunk:
            continue

        if used_chars + len(chunk) > max_context_chars:
            continue

        chosen_ids.append(idx)
        used_chars += len(chunk)

    # Restore document order for easier interpretation.
    chosen_ids.sort()

    context = "\n\n".join(
        pdf_chunks[idx] for idx in chosen_ids
    )

    if not context:
        print("No relevant document context found.")
        return

    # STEP 6: Prepare the general-purpose answer prompt.
    messages = [
        {
            "role": "system",
            "content": (
                "Answer using only the supplied document excerpts. "
                "Give a direct and accurate answer. Include all "
                "relevant items when asked for a list or count. "
                "Preserve names, dates, numbers and units exactly. "
                "Do not invent information or infer facts that the "
                "document does not support. If the answer is absent, "
                "say 'Not specified in the document.' Return only "
                "the answer, without source labels."
            )
        },
        {
            "role": "user",
            "content": (
                f"Document excerpts:\n{context}\n\n"
                f"Question: {question}"
            )
        }
    ]

    # STEP 7: Format the prompt for Qwen.
    prompt = generator.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # STEP 8: Generate the answer on the GPU.
    result = generator(
        prompt,
        max_new_tokens=200,
        do_sample=False,
        return_full_text=False,
        pad_token_id=generator.tokenizer.eos_token_id
    )

    answer = result[0]["generated_text"].strip()
    print(answer if answer else "No answer generated.")


print("Updated RAG function is ready!")

Updated RAG function is ready!


# Step 1 — Test your sample PDF

In [ ]:
ask_pdf("How many accounts were in the evaluation dataset, and how many cancellations were recorded?")

24,000 accounts, 3,120 cancellations in the evaluation set.


In [ ]:
ask_pdf("List all project milestone dates and planned future dates.")

6 Jan 2025 — Project kickoff and stakeholder interviews
31 Jan 2025 — Data inventory and target definition approved
21 Feb 2025 — Data quality checks and initial feature list
28 Mar 2025 — Baseline model comparison
25 Apr 2025 — Threshold review and error analysis
23 May 2025 — API prototype and request validation
20 Jun 2025 — Pilot operating guide and risk review
Oct-Nov 2025 — Phase 1: Controlled pilot
6 Oct 2025 — First planned operational use; Planned start of controlled pilot
7 Nov 2025 — Formal pilot review and go/no-go decision
Dec 2025 — Phase 2: Evaluation
Jan 2026 — Phase 3: Workflow refinement
Feb 2026 — Phase 4: Broader rollout decision


In [ ]:
ask_pdf("Which model was selected, and what were its ROC AUC and precision?")

The selected model was Gradient-Boosted Trees, which achieved an ROC AUC of 0.81 and a precision of 0.68 at the approved decision threshold.


In [ ]:
ask_pdf("What is the approved churn-risk threshold and prediction window?")

The approved churn-risk threshold is a churn probability of 0.62. The prediction window is the 30 days after the scoring date.


In [ ]:
# ============================================================
# FINAL RAG FUNCTION
# Timeline extraction + general PDF question answering
# ============================================================

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


def ask_pdf(question):
    if not question or not question.strip():
        print("Please enter a question.")
        return

    q = question.lower().strip()

    # STEP 1: Route only timeline-related questions.
    timeline_terms = (
        "timeline",
        "milestone",
        "milestones",
        "roadmap",
        "planned future",
        "pilot dates",
        "rollout phases",
        "project dates",
        "project milestone dates",
    )

    if any(term in q for term in timeline_terms):
        entries = extract_document_dates(pdf_text)

        if entries:
            print("\n".join(entries))
        else:
            print("No standalone dates or milestones found.")
        return

    # STEP 2: Embed the question for general RAG retrieval.
    query_vector = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)[0]

    # STEP 3: Semantic similarity search.
    semantic_scores = pdf_embeddings @ query_vector
    score_range = np.ptp(semantic_scores)

    if score_range > 1e-8:
        semantic_scores = (
            semantic_scores - semantic_scores.min()
        ) / score_range
    else:
        semantic_scores = np.zeros_like(semantic_scores)

    # STEP 4: Keyword similarity search.
    lexical_scores = cosine_similarity(
        tfidf_vectorizer.transform([question]),
        tfidf_matrix
    ).ravel()

    # STEP 5: Combine search scores.
    combined_scores = (
        0.60 * semantic_scores
        + 0.40 * lexical_scores
    )

    ranked_ids = np.argsort(combined_scores)[::-1]
    selected_ids = set()

    # Include nearby chunks to preserve context.
    for idx in ranked_ids[:10]:
        idx = int(idx)

        for neighbor in (idx - 1, idx, idx + 1):
            if 0 <= neighbor < len(pdf_chunks):
                selected_ids.add(neighbor)

    # Keep the best-ranked chunks within the context budget.
    prioritized_ids = sorted(
        selected_ids,
        key=lambda i: combined_scores[i],
        reverse=True
    )

    chosen_ids = []
    used_chars = 0
    max_context_chars = 6000

    for idx in prioritized_ids:
        chunk = pdf_chunks[idx].strip()

        if not chunk:
            continue

        if used_chars + len(chunk) > max_context_chars:
            continue

        chosen_ids.append(idx)
        used_chars += len(chunk)

    # Restore document order for easier interpretation.
    chosen_ids.sort()

    context = "\n\n".join(
        pdf_chunks[idx] for idx in chosen_ids
    )

    if not context:
        print("No relevant document context found.")
        return

    # STEP 6: Prepare the general-purpose answer prompt.
    messages = [
        {
            "role": "system",
            "content": (
                "Answer using only the supplied document excerpts. "
                "Give a direct and accurate answer. Include all "
                "relevant items when asked for a list or count. "
                "Preserve names, dates, numbers and units exactly. "
                "Do not invent information or infer facts that the "
                "document does not support. If the answer is absent, "
                "say 'Not specified in the document.' Return only "
                "the answer, without source labels."
            )
        },
        {
            "role": "user",
            "content": (
                f"Document excerpts:\n{context}\n\n"
                f"Question: {question}"
            )
        }
    ]

    # STEP 7: Format the prompt for Qwen.
    prompt = generator.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )ask_pdf("What is the CEO's date of birth?")

    # STEP 8: Generate the answer on the GPU.
    result = generator(
        prompt,
        max_new_tokens=200,
        do_sample=False,
        return_full_text=False,
        pad_token_id=generator.tokenizer.eos_token_id
    )

    answer = result[0]["generated_text"].strip()
    print(answer if answer else "No answer generated.")


print("Updated RAG function is ready!")

Updated RAG function is ready!


In [ ]:
ask_pdf("What is the CEO's date of birth?")

Not specified in the document.


In [ ]:
ask_pdf("List all project milestone dates and planned future dates.")

6 Jan 2025 — Project kickoff and stakeholder interviews
31 Jan 2025 — Data inventory and target definition approved
21 Feb 2025 — Data quality checks and initial feature list
28 Mar 2025 — Baseline model comparison
25 Apr 2025 — Threshold review and error analysis
23 May 2025 — API prototype and request validation
20 Jun 2025 — Pilot operating guide and risk review
Oct-Nov 2025 — Phase 1: Controlled pilot
6 Oct 2025 — First planned operational use; Planned start of controlled pilot
7 Nov 2025 — Formal pilot review and go/no-go decision
Dec 2025 — Phase 2: Evaluation
Jan 2026 — Phase 3: Workflow refinement
Feb 2026 — Phase 4: Broader rollout decision
